<a href="https://colab.research.google.com/github/Lourdes-Churacutipa/Repositorio-de-lourdes/blob/main/PC3_precipitacion_Santiago_Tuna_share.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PC3 – Evaluación de Productos de Precipitación en la Sierra de Lima
## Estación Santiago de Tuna · Huarochirí · 2926 msnm

| | |
| --- | --- |
| **Curso** | AM4314 – Ciencias de Datos Ambientales |
| **Ciclo** | 2026-I |
| **Temas** | 5.1 Series de tiempo · 5.2 Datos grillados/NetCDF · 6.1 GEE fundamentos |
| **Puntaje total** | 20 puntos |
| **Entrega** | Notebook ejecutado (.ipynb) + precipitacion_Santiago_Tuna.csv |

## Enunciado del problema

### Contexto científico

La cuenca alta del río Rímac abastece de agua potable a más del 30 % de la población del Perú (Lima Metropolitana). Monitorear la precipitación en esta cuenca es crítico para la gestión de reservorios, alertas de inundaciones y planificación ante el cambio climático.

Las redes de estaciones meteorológicas en los Andes son escasas y presentan vacíos temporales. Por ello, los hidrólogos recurren a **productos de precipitación grillados** para reconstruir la variabilidad de la lluvia. Sin embargo, cada producto tiene sesgos propios en zonas de relieve complejo y alta altitud.

### Pregunta de investigación

¿Qué producto de precipitación grillado (PISCOp o ERA5-Land) representa mejor la precipitación observada en la estación Santiago de Tuna (2926 msnm), y cómo se relaciona la precipitación local con el fenómeno El Niño Costero?

### Datos disponibles

| Fuente | Tipo | Período | Resolución |
| --- | --- | --- | --- |
| Estación Santiago de Tuna (472CA750) | Observación puntual (ground truth) | 1964–2014 | Diaria |
| PISCOp (SENAMHI) | Grillado nacional calibrado | 1981–2025 | ~5 km, mensual |
| ERA5-Land (ECMWF) | Reanálisis global | 1950–presente | ~9 km, mensual |
| ICEN (IGP) | Índice Costero El Niño | 1950–presente | Mensual |

### Tareas

1. Cargar los datos observados de la estación y construir una serie mensual de referencia.
2. Extraer la serie mensual de PISCOp en el punto de la estación desde un archivo NetCDF.
3. Extraer la serie mensual de ERA5-Land desde Google Earth Engine.
4. Integrar las tres fuentes en un DataFrame ordenado y exportarlo a CSV.
5. Descargar el índice ICEN del IGP y calcular su correlación con la precipitación observada.
6. Generar gráficos de series de tiempo, diagramas de dispersión y mapa de correlaciones.
7. Evaluar el desempeño de cada producto con métricas estadísticas (r, RMSE, Bias) y concluir.

### Rúbrica

| Sección | Descripción | Puntos |
| --- | --- | --- |
| P0 | Cargar datos observados y obtener serie mensual | 2 |
| P1 | Cargar PISCOp (NetCDF) y extraer serie en el punto | 3 |
| P2 | Extraer ERA5-Land desde GEE + conversión de unidades | 4 |
| P3 | DataFrame unificado (OBS + PISCO + ERA5-Land) y exportar CSV | 2 |
| P4 | Cargar índice ICEN y correlación con OBS | 3 |
| P5 | Visualizaciones: series de tiempo, scatter, heatmap | 4 |
| P6 | Métricas (r, RMSE, Bias) vs OBS y ranking | 2 |
| **Total** | | **20** |

---
## Configuración inicial

Ejecuta las siguientes celdas sin modificar.

In [1]:
# ── Librerías ─────────────────────────────────────────────────────────────────
import ee
import geemap
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import requests
import gdown
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Librerías cargadas ✓")

Mounted at /content/drive
Librerías cargadas ✓


In [3]:
ee.Authenticate()
ee.Initialize(project='ee-lchuracutipa')   # ← reemplaza con tu proyecto GEE
print("GEE inicializado ✓")

GEE inicializado ✓


In [ ]:
# ── Parámetros de la estación ─────────────────────────────────────────────────
NOMBRE      = 'Santiago de Tuna'
LAT         = -11.9832    # 11°58'59.6" S
LON         = -76.5242    # 76°31'27.1" W
ALT         = 2926        # msnm
CODIGO      = '472CA750'

PISCO_FILE_ID = '13FvNPV6nCWspgFxG1cBYhem1n2tUKXH6'
OBS_FILE_ID   = '1EKemYnWTc5W-oSv7gBCrcxCKhOYA7nEV'

INICIO = '1981-01-01'
FIN    = '2014-12-31'   # serie observada disponible hasta 2014

print(f"Estación : {NOMBRE}  [{CODIGO}]")
print(f"Lat/Lon  : {LAT}°, {LON}°  |  Alt: {ALT} msnm")
print(f"Período  : {INICIO[:7]} → {FIN[:7]}")

Estación : Santiago de Tuna  [472CA750]
Lat/Lon  : -11.9832°, -76.5242°  |  Alt: 2926 msnm
Período  : 1981-01 → 2014-12


---
## P0 – Datos observados de la estación (2 pts)

Los datos de Santiago de Tuna son el **ground truth**. Todos los productos se evalúan contra esta serie.

Formato del archivo: sin cabecera, separado por espacios.
Columnas: Año | Mes | Día | Precipitación acumulada (mm) | Tmax | Tmin
Valor faltante: -99.9

| Sub-tarea | Puntos |
| --- | --- |
| P0.1 Descargar el archivo con gdown | 0.5 |
| P0.2 Cargar con pandas y aplicar la estructura de columnas correcta | 0.5 |
| P0.3 Construir serie mensual con índice MS y nombre 'OBS' | 1.0 |

In [ ]:
# TU CÓDIGO AQUÍ — gdown.download con OBS_FILE_ID, guardar como 'datos_obs.txt'


In [ ]:
# TU CÓDIGO AQUÍ — pd.read_csv con sep='\\s+', header=None, names=[...]; reemplazar -99.9 por NaN


In [ ]:
# TU CÓDIGO AQUÍ — construir fecha desde anio/mes/dia; resample('MS').sum(min_count=15); nombre 'OBS'


---
## P1 – Datos PISCOp (3 pts)

PISCOp es el producto de referencia de SENAMHI, calibrado con estaciones peruanas.

| Sub-tarea | Puntos |
| --- | --- |
| P1.1 Cargar NetCDF con xarray y mostrar estructura | 0.5 |
| P1.2 Identificar la variable de precipitación y sus unidades | 0.5 |
| P1.3 Extraer serie temporal en el punto con method='nearest' | 1.0 |
| P1.4 Convertir a pd.Series con índice MS y nombre 'PISCO' | 1.0 |

In [ ]:
# P1.1 – Descargar y cargar PISCOp (NetCDF)
# Pasos: (1) gdown.download con PISCO_FILE_ID → 'pisco.nc'
#        (2) xr.open_dataset('pisco.nc')
#        (3) renombrar dimensión temporal 'Z1' → 'time'
#        (4) asignar pd.date_range(start='1981-01-01', periods=..., freq='MS') como coordenada time
# TU CÓDIGO AQUÍ


In [ ]:
# P1.2 – Identificar variable de precipitación
# La variable se llama 'precipitation'; asigna: VAR_PISCO = 'precipitation'
# TU CÓDIGO AQUÍ


In [ ]:
# TU CÓDIGO AQUÍ — ds_pisco[VAR].sel(lat=LAT, lon=LON, method='nearest')


In [ ]:
# TU CÓDIGO AQUÍ — convertir a pd.Series con índice MS y nombre 'PISCO'


---
## P2 – ERA5-Land desde Google Earth Engine (4 pts)

ERA5-Land (ECMWF) es un reanálisis global de alta resolución accesible directamente desde GEE.

| Colección GEE | Banda | Resolución | Unidades originales |
| --- | --- | --- | --- |
| ECMWF/ERA5_LAND/MONTHLY_AGGR | total_precipitation_sum | ~9 km | m/mes → x1000 para mm |

| Sub-tarea | Puntos |
| --- | --- |
| P2.1 Crear punto GEE y filtrar la colección por fecha | 1.0 |
| P2.2 Extraer serie mensual con gee_monthly_ts | 2.0 |
| P2.3 Verificar unidades y estadísticas | 1.0 |

In [4]:
# ── Función auxiliar GEE (proporcionada — no modificar) ──────────────────────
point = ee.Geometry.Point([LON, LAT])

def gee_monthly_ts(collection, band, point, scale, nombre, multiplicador=1.0):
    """Extrae serie temporal mensual de un ee.ImageCollection en un punto."""
    def sample(img):
        val = img.select(band).reduceRegion(
            reducer=ee.Reducer.first(), geometry=point, scale=scale
        ).get(band)
        return ee.Feature(None, {
            'date' : img.date().format('YYYY-MM-dd'),
            'value': ee.Number(val).multiply(multiplicador)
        })
    data = ee.FeatureCollection(collection.map(sample)).getInfo()['features']
    df = pd.DataFrame([
        {'date': f['properties']['date'], 'value': f['properties']['value']}
        for f in data
    ])
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    return pd.Series(
        df['value'].values,
        index=pd.DatetimeIndex(df['date']).to_period('M').to_timestamp(),
        name=nombre
    ).asfreq('MS')

print("Función gee_monthly_ts lista ✓")
print(f"Punto GEE: ({LON}, {LAT})")

NameError: name 'LON' is not defined

In [ ]:
# TU CÓDIGO AQUÍ — filtrar ERA5-Land: .filterDate(INICIO, FIN).select('total_precipitation_sum')


In [ ]:
# TU CÓDIGO AQUÍ — gee_monthly_ts con multiplicador=1000


In [ ]:
# TU CÓDIGO AQUÍ — verificar estadísticas: describe()


---
## P3 – DataFrame unificado y exportación a CSV (2 pts)

Columnas esperadas: OBS | PISCO | ERA5-Land

| Sub-tarea | Puntos |
| --- | --- |
| P3.1 Unir las 3 series alineadas por fecha | 1.0 |
| P3.2 Estadísticas, valores faltantes y exportar CSV | 1.0 |

In [ ]:
# TU CÓDIGO AQUÍ — pd.concat([obs_series, pisco_series, era5_series], axis=1)


In [ ]:
# TU CÓDIGO AQUÍ — describe(), isna().sum(), to_csv(...)


---
## P4 – Índice ICEN y correlación con precipitación observada (3 pts)

El [ICEN](http://met.igp.gob.pe/datos/ICEN.txt) (Índice Costero El Niño, IGP) mide anomalías de TSM en la región Niño 1+2 costera.
Valores positivos = El Niño; negativos = La Niña.

| ICEN | Categoría |
| --- | --- |
| mayor o igual a 1.7 | El Niño extraordinario |
| 1.0 – 1.6 | El Niño fuerte |
| 0.4 – 0.9 | El Niño moderado |
| −0.3 – 0.3 | Normal |
| −0.4 a −0.9 | La Niña moderada |
| menor o igual a −1.0 | La Niña fuerte |

| Sub-tarea | Puntos |
| --- | --- |
| P4.1 Descargar y parsear el archivo ICEN del IGP | 1.0 |
| P4.2 Unir al DataFrame; correlación Pearson con OBS | 1.0 |
| P4.3 Scatter OBS vs ICEN coloreado por categoría ENSO | 1.0 |

In [ ]:
# TU CÓDIGO AQUÍ — requests.get(URL_ICEN) + mostrar primeras líneas


In [ ]:
# TU CÓDIGO AQUÍ — parsear ICEN y hacer df = df.join(icen_df, how='left')


In [ ]:
# TU CÓDIGO AQUÍ — stats.pearsonr(df['ICEN'], df['OBS']) — solo meses sin NaN


In [ ]:
# TU CÓDIGO AQUÍ — scatter OBS vs ICEN coloreado por categoría ENSO


---
## P5 – Visualizaciones (4 pts)

| Sub-tarea | Puntos |
| --- | --- |
| P5.1 Serie de tiempo: OBS, PISCO y ERA5-Land (1981–2014) | 1.5 |
| P5.2 Scatter de cada producto vs OBS con línea 1:1 y regresión | 1.5 |
| P5.3 Heatmap de correlaciones entre OBS, PISCO, ERA5-Land e ICEN | 1.0 |

In [ ]:
# TU CÓDIGO AQUÍ — plot OBS + PISCO + ERA5-Land en el mismo eje temporal


In [ ]:
# TU CÓDIGO AQUÍ — 2 scatter (PISCO vs OBS y ERA5-Land vs OBS) con línea 1:1


In [ ]:
# TU CÓDIGO AQUÍ — heatmap de correlaciones entre OBS, PISCO, ERA5-Land, ICEN


---
## P6 – Métricas de evaluación y ranking (2 pts)

Para cada producto respecto a OBS:
- **r** — Correlación de Pearson (ideal = 1)
- **RMSE** — Raíz del error cuadrático medio en mm/mes (ideal = 0)
- **Bias** — media(producto) − media(OBS) en mm/mes (ideal = 0)

| Sub-tarea | Puntos |
| --- | --- |
| P6.1 Calcular r, RMSE y Bias para PISCO y ERA5-Land | 1.0 |
| P6.2 Tabla resumen y ranking | 1.0 |

In [5]:
# Función de métricas (proporcionada)
def metricas(df, sim_col, obs_col='OBS'):
    tmp = df[[obs_col, sim_col]].dropna()
    o, s = tmp[obs_col].values, tmp[sim_col].values
    r, _  = stats.pearsonr(o, s)
    rmse  = np.sqrt(np.mean((s-o)**2))
    bias  = s.mean() - o.mean()
    return {'r': round(r,3), 'RMSE (mm/mes)': round(rmse,1),
            'Bias (mm/mes)': round(bias,1), 'N meses': len(tmp)}

In [ ]:
# TU CÓDIGO AQUÍ — metricas(df, 'PISCO') y metricas(df, 'ERA5-Land') → tabla


In [ ]:
# TU CÓDIGO AQUÍ — tabla.sort_values('RMSE (mm/mes)') → ranking


---
## P7 – Conclusiones (2 pts)

Responde en 200–350 palabras apoyando cada afirmación con métricas o gráficos.

1. **Mejor producto**: ¿PISCOp o ERA5-Land? Justifica con r, RMSE y Bias.
2. **Limitaciones por altitud**: dificultades de los grillados a 2926 msnm.
3. **Relación ENSO–precipitación**: signo de la correlación OBS–ICEN y mecanismo físico.
4. **Fases ENSO extremas**: en qué categorías se registran máximos y mínimos de precipitación.

### Tus conclusiones

*(Reemplaza este texto — 200–350 palabras)*

**1. Mejor producto:** ...

**2. Limitaciones por altitud:** ...

**3. Relación ENSO–precipitación:** ...

**4. Fases ENSO extremas:** ...